# SkyGuard AI: Exploratory Data Analysis & Feature Engineering Handoff
**Smart India Hackathon 2026 | Problem Statement 26073 | Disaster Management**

This notebook provides an interactive, physics-informed exploratory analysis of Automatic Weather Station (AWS) telemetry.
It is parameterized to accept either synthetic datasets or real Open-Meteo / IMD AWS pulls (`temperature_c`, `pressure_hpa`, `humidity_pct`).

In [ ]:
# Parameters
DATA_PATH = '../data/raw/aws_telemetry_master.parquet'
OUTPUT_DIR = '../data/eda_plots'
REPORT_PATH = '../docs/EDA_INSIGHTS.md'
DPI = 200

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
from eda_pipeline import SkyGuardEDA

print('Initializing SkyGuardEDA pipeline...')

## 1. Execute Complete Pipeline or Step-by-Step
We initialize the analyzer object pointing to `DATA_PATH`.

In [ ]:
eda = SkyGuardEDA(input_path=DATA_PATH, output_dir=OUTPUT_DIR, report_path=REPORT_PATH, dpi=DPI)
eda.df.head()

## 2. Data Overview & Missingness
Examines station sampling rates, null distributions, and binary/multiclass balance.

In [ ]:
eda.run_overview()
display(Image(filename=os.path.join(OUTPUT_DIR, '01_data_overview_missingness.png')))

## 3. Univariate Distributions & Physical Bounds
Validates physical envelopes and identifies data corruption sentinel codes (-999, 999).

In [ ]:
eda.run_univariate()
display(Image(filename=os.path.join(OUTPUT_DIR, '02_univariate_distributions.png')))

## 4. Temporal Dynamics, STL & Autoregressive Memory (GRU Window Sizing)
Performs seasonal decomposition (STL) and ACF/PACF analysis.
**Key Insight**: Sharp diurnal peak at lag 144 (24h) and S2 thermal tide at lag 72 (12h) dictate the optimal GRU window.

In [ ]:
eda.run_temporal()
display(Image(filename=os.path.join(OUTPUT_DIR, '03_temporal_stl_acf_pacf.png')))

## 5. Multivariate & Thermodynamic Consistency (Mahalanobis Tier)
Demonstrates that cross-sensor inconsistencies violate the joint covariance manifold ($D_M > 7.0$ vs normal 1.6).

In [ ]:
eda.run_multivariate()
display(Image(filename=os.path.join(OUTPUT_DIR, '04_multivariate_thermodynamic_mahalanobis.png')))

## 6. Spatial Cross-Station Correlation (Buddy-Check Validation)
Validates that neighboring stations move together during severe weather, but diverge 6x during sensor hardware failures.

In [ ]:
eda.run_spatial()
display(Image(filename=os.path.join(OUTPUT_DIR, '05_spatial_buddy_correlation.png')))

## 7. Anomaly-Type Profiling & Separability Taxonomy
Profiles statistical footprints (step jump, zero variance flatline, drift slope, volatility surge).

In [ ]:
eda.run_anomaly_profiling()
display(Image(filename=os.path.join(OUTPUT_DIR, '06_anomaly_type_footprints.png')))

## 8. Naive Baseline Benchmark Floor
Demonstrates the blindspots of Z-score and IQR baselines (0% recall on frozen sensors and joint outliers).

In [ ]:
eda.run_baseline_qc()
display(Image(filename=os.path.join(OUTPUT_DIR, '07_baseline_qc_benchmark.png')))

## 9. Compiled Insights Handoff Report

In [ ]:
eda.compile_report()
with open(REPORT_PATH, 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))